# Week 15 과제 · 모범답안 (교수자용)

> 교수자용 모범답안입니다. 학생에게 배포하지 마세요.

## 준비: 데이터와 자동 확인 도우미

In [1]:
import sys
import hashlib
from pathlib import Path
import urllib.request

import numpy as np
import pandas as pd
import matplotlib

DATA_URL = "https://raw.githubusercontent.com/MNU-AI-Programming/AI-Assisted-Python-Programming/main/datasets/"

def get_data(name):
    for folder in [Path("../../datasets"), Path("datasets"), Path(".")]:
        if (folder / name).exists():
            return str(folder / name)
    urllib.request.urlretrieve(DATA_URL + name, name)
    return name

STUDENTS = get_data("students.csv")
CAFE = pd.read_csv(get_data("cafe_sales.csv"), parse_dates=["date"])


def check(desc, test):
    """자동 확인 도우미: test는 True/False를 돌려주는 함수(lambda)입니다."""
    try:
        ok = test()
    except NotImplementedError:
        print(f"⏳ 미작성 - {desc}")
    except Exception as e:
        print(f"❌ 오류 - {desc} ({type(e).__name__}: {e})")
    else:
        print(f"{'✅ 통과' if ok else '❌ 실패'} - {desc}")

---
## 문제 1. 환경과 데이터 지문 기록 (2점)

1. `env_info()`: `{"python": "3.x.y", "numpy": ..., "pandas": ..., "matplotlib": ...}` – 모두 **버전 문자열**
2. `data_fingerprint(path)`: CSV 파일의 지문 딕셔너리
   - `"rows"`: 헤더를 뺀 데이터 행 수 (pandas로 읽은 행 수)
   - `"cols"`: 열 이름 리스트
   - `"md5"`: 파일 **바이트 전체**의 MD5 해시 16진수 문자열 앞 **12자리** (`hashlib.md5(...).hexdigest()`)

데이터 파일이 한 글자만 바뀌어도 해시가 달라지므로, 결과가 달라졌을 때 **데이터가 같은지** 바로 확인할 수 있습니다.

In [2]:
def env_info():
    return {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": matplotlib.__version__,
    }


def data_fingerprint(path):
    raw = Path(path).read_bytes()
    df = pd.read_csv(path)
    return {"rows": len(df), "cols": list(df.columns), "md5": hashlib.md5(raw).hexdigest()[:12]}

In [3]:
EXPECTED_MD5 = "d8be6c0c1b8e"            # students.csv 기준값


def _changed_hash_differs():
    import tempfile
    p = Path(tempfile.mkdtemp()) / "copy.csv"
    p.write_bytes(Path(STUDENTS).read_bytes().replace(b"100", b"99", 1))
    return data_fingerprint(str(p))["md5"] != data_fingerprint(STUDENTS)["md5"]


check("env_info 키", lambda: list(env_info()) == ["python", "numpy", "pandas", "matplotlib"])
check("버전은 문자열", lambda: all(isinstance(v, str) and v[0].isdigit() for v in env_info().values()))
check("python 버전 일치", lambda: env_info()["python"] == f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
check("students.csv 행·열", lambda: data_fingerprint(STUDENTS)["rows"] == 80 and data_fingerprint(STUDENTS)["cols"][:2] == ["student_id", "name"])
check("students.csv md5", lambda: data_fingerprint(STUDENTS)["md5"] == EXPECTED_MD5)
check("한 글자만 바뀌어도 해시가 다름", _changed_hash_differs)

✅ 통과 - env_info 키
✅ 통과 - 버전은 문자열
✅ 통과 - python 버전 일치
✅ 통과 - students.csv 행·열
✅ 통과 - students.csv md5
✅ 통과 - 한 글자만 바뀌어도 해시가 다름


---
## 문제 2. 결과표를 Markdown으로 (2점)

README와 보고서에 결과표를 붙이기 위해 DataFrame을 Markdown 표 문자열로 바꾸는 `to_markdown_table(df, digits=1)`를 만드세요. (외부 라이브러리 `tabulate` 없이)

- 첫 줄: `| 열1 | 열2 |`, 둘째 줄: 열마다 `---` → `| --- | --- |`
- 값: 실수는 소수 `digits`자리 (`f"{v:.{digits}f}"`), 정수는 천 단위 쉼표 (`f"{v:,}"`), 그 외는 `str(v)`
- 인덱스는 넣지 않습니다. 줄은 `"\n"`으로 연결하고 마지막 줄 뒤에는 줄바꿈이 없습니다.

```python
to_markdown_table(pd.DataFrame({"메뉴": ["아메리카노"], "매출": [3352700], "비중": [23.61]}))
# "| 메뉴 | 매출 | 비중 |\n| --- | --- | --- |\n| 아메리카노 | 3,352,700 | 23.6 |"
```

In [4]:
def to_markdown_table(df, digits=1):
    def fmt(v):
        if isinstance(v, (bool, np.bool_)):
            return str(v)
        if isinstance(v, (int, np.integer)):
            return f"{v:,}"
        if isinstance(v, (float, np.floating)):
            return f"{v:.{digits}f}"
        return str(v)

    lines = ["| " + " | ".join(str(c) for c in df.columns) + " |",
             "| " + " | ".join("---" for _ in df.columns) + " |"]
    for row in df.itertuples(index=False):
        lines.append("| " + " | ".join(fmt(v) for v in row) + " |")
    return "\n".join(lines)

In [5]:
t1 = pd.DataFrame({"메뉴": ["아메리카노"], "매출": [3352700], "비중": [23.61]})
check("예시", lambda: to_markdown_table(t1) == "| 메뉴 | 매출 | 비중 |\n| --- | --- | --- |\n| 아메리카노 | 3,352,700 | 23.6 |")
check("digits=2", lambda: to_markdown_table(t1, digits=2).endswith("| 23.61 |"))
t2 = pd.DataFrame({"a": [1, 2], "b": ["x", "y"]}, index=[10, 20])
check("인덱스 제외, 여러 행", lambda: to_markdown_table(t2).splitlines() == ["| a | b |", "| --- | --- |", "| 1 | x |", "| 2 | y |"])
check("마지막 줄바꿈 없음", lambda: not to_markdown_table(t2).endswith("\n"))

✅ 통과 - 예시
✅ 통과 - digits=2
✅ 통과 - 인덱스 제외, 여러 행
✅ 통과 - 마지막 줄바꿈 없음


In [6]:
# 실제 결과표로 확인
menu = (CAFE.assign(revenue=CAFE["unit_price"] * CAFE["qty"]).groupby("menu")["revenue"].sum()
            .sort_values(ascending=False).head(3).reset_index())
menu["share"] = menu["revenue"] / (CAFE["unit_price"] * CAFE["qty"]).sum() * 100
try:
    print(to_markdown_table(menu))
except NotImplementedError:
    print("아직 작성하지 않았습니다.")

| menu | revenue | share |
| --- | --- | --- |
| 아메리카노 | 3,352,700 | 23.6 |
| 카페라떼 | 2,643,300 | 18.6 |
| 바닐라라떼 | 1,381,600 | 9.7 |


---
## 문제 3. 시드가 있는 부트스트랩 신뢰구간 (3점)

표본이 작을 때 평균이 얼마나 흔들리는지 보여 주는 **부트스트랩 95% 신뢰구간**을 만드세요. 시드를 고정하면 몇 번을 실행해도 같은 구간이 나와야 합니다(재현성).

`bootstrap_ci(values, n_boot=2000, seed=42)`

1. `rng = np.random.default_rng(seed)`
2. `n_boot`번 반복: `values`에서 **같은 개수를 복원 추출**(`rng.choice(values, size=len(values), replace=True)`)해 평균을 구한다
3. 평균들의 2.5%, 97.5% 백분위수(`np.percentile`)를 소수 첫째 자리로 반올림해 `(하한, 상한)` 튜플로 돌려준다
4. 값이 2개 미만이면 `ValueError`

그리고 `exam_vs_normal_ci(cafe)`로 **시험 주간 / 평상시(평일)** 일별 주문 건수의 신뢰구간을 각각 구해 `{"시험 주간": (하한, 상한), "평상시": (하한, 상한)}`로 돌려주세요.
시험 주간: 2025-04-21~25, 2025-06-16~20 (월~금). 평상시: 그 외 **평일(월~금)**.

In [7]:
def bootstrap_ci(values, n_boot=2000, seed=42):
    values = np.asarray(values, dtype=float)
    if len(values) < 2:
        raise ValueError("값이 2개 이상 필요합니다")
    rng = np.random.default_rng(seed)
    means = [rng.choice(values, size=len(values), replace=True).mean() for _ in range(n_boot)]
    low, high = np.percentile(means, [2.5, 97.5])
    return round(float(low), 1), round(float(high), 1)


def exam_vs_normal_ci(cafe):
    orders = cafe.groupby("date").size()
    dates = orders.index
    exam = pd.Series(False, index=dates)
    for start in ["2025-04-21", "2025-06-16"]:
        s = pd.Timestamp(start)
        exam |= (dates >= s) & (dates <= s + pd.Timedelta(days=4))
    weekday = dates.dayofweek < 5
    return {"시험 주간": bootstrap_ci(orders[exam & weekday].values),
            "평상시": bootstrap_ci(orders[~exam & weekday].values)}

In [8]:
EXPECTED_SAMPLE_CI = (4.6, 9.4)                                         # 기준값
EXPECTED_EXAM_CI = {"시험 주간": (38.3, 43.6), "평상시": (32.7, 35.6)}     # cafe_sales.csv 기준값


def _raises_value_error(fn, *args):
    try:
        fn(*args)
    except ValueError:
        return True
    return False


sample = [3, 5, 7, 9, 11]
check("같은 시드 → 같은 구간", lambda: bootstrap_ci(sample, seed=1) == bootstrap_ci(sample, seed=1))
check("구간이 평균을 포함", lambda: (lambda lo_hi: lo_hi[0] <= 7 <= lo_hi[1])(bootstrap_ci(sample)))
check("값 1개 → ValueError", lambda: _raises_value_error(bootstrap_ci, [5]))
check("기준값 (sample, seed=42)", lambda: bootstrap_ci(sample) == EXPECTED_SAMPLE_CI)
check("시험 주간 vs 평상시", lambda: exam_vs_normal_ci(CAFE) == EXPECTED_EXAM_CI)
check("시험 주간 구간이 더 넓다 (표본이 작다)", lambda: (lambda r: (r["시험 주간"][1] - r["시험 주간"][0]) > (r["평상시"][1] - r["평상시"][0]))(exam_vs_normal_ci(CAFE)))

✅ 통과 - 같은 시드 → 같은 구간
✅ 통과 - 구간이 평균을 포함
✅ 통과 - 값 1개 → ValueError
✅ 통과 - 기준값 (sample, seed=42)
✅ 통과 - 시험 주간 vs 평상시


✅ 통과 - 시험 주간 구간이 더 넓다 (표본이 작다)


> 💡 시험 주간은 10일뿐이라 신뢰구간이 넓습니다. "시험 주간에 주문이 늘었다"는 결론을 쓸 때 **구간이 겹치는지**를 함께 보면, 결론을 얼마나 강하게 말해도 되는지 판단할 수 있습니다.

---
## 문제 4. README 생성기와 내 프로젝트 README (AI 협업, 3점)

### 4-1. `make_readme(title, summary, questions, data_rows, run_steps, limits, ai_note)`

다음 **절 순서**를 가진 README 문자열을 돌려줍니다.

```text
# {title}

> {summary}

## 연구 질문
1. 질문1
2. 질문2

## 데이터
| 파일 | 출처 | 기간 |
| --- | --- | --- |
| ... | ... | ... |

## 실행 방법
1. 단계1
2. 단계2

## 한계
- 한계1
- 한계2

## AI 활용
{ai_note}
```

- `questions`, `run_steps`, `limits`는 문자열 리스트, `data_rows`는 `(파일, 출처, 기간)` 튜플 리스트입니다.
- 절과 절 사이에는 **빈 줄 하나**, 문자열 끝에는 줄바꿈 하나.
- `questions`가 비어 있으면 `ValueError` (연구 질문 없는 프로젝트는 없다!)

### 4-2. 내 최종 프로젝트 README 초안

`make_readme`로 **여러분의 최종 프로젝트** README 초안을 만들어 출력하세요. AI에게 초안 문장을 도움받았다면 아래 기록에 남깁니다.

In [9]:
def make_readme(title, summary, questions, data_rows, run_steps, limits, ai_note):
    if not questions:
        raise ValueError("연구 질문이 필요합니다")
    sections = [
        f"# {title}",
        f"> {summary}",
        "## 연구 질문\n" + "\n".join(f"{i}. {q}" for i, q in enumerate(questions, 1)),
        "## 데이터\n| 파일 | 출처 | 기간 |\n| --- | --- | --- |\n"
        + "\n".join(f"| {f} | {s} | {p} |" for f, s, p in data_rows),
        "## 실행 방법\n" + "\n".join(f"{i}. {s}" for i, s in enumerate(run_steps, 1)),
        "## 한계\n" + "\n".join(f"- {x}" for x in limits),
        f"## AI 활용\n{ai_note}",
    ]
    return "\n\n".join(sections) + "\n"

In [10]:
_args = dict(title="카페 분석", summary="점심이 바쁘다", questions=["점심 비중은?", "기온 효과는?"],
             data_rows=[("cafe_sales.csv", "수업 데이터", "2025-03~06")], run_steps=["Colab에서 연다", "모두 실행"],
             limits=["가상 데이터"], ai_note="그래프 초안에 사용, 직접 수정")
check("제목·요약", lambda: make_readme(**_args).startswith("# 카페 분석\n\n> 점심이 바쁘다\n\n## 연구 질문\n1. 점심 비중은?\n2. 기온 효과는?"))
check("절 순서", lambda: [l for l in make_readme(**_args).splitlines() if l.startswith("## ")]
      == ["## 연구 질문", "## 데이터", "## 실행 방법", "## 한계", "## AI 활용"])
check("데이터 표", lambda: "| 파일 | 출처 | 기간 |\n| --- | --- | --- |\n| cafe_sales.csv | 수업 데이터 | 2025-03~06 |" in make_readme(**_args))
check("끝 줄바꿈 하나", lambda: make_readme(**_args).endswith("직접 수정\n") and not make_readme(**_args).endswith("\n\n"))

def _no_question_raises():
    try:
        make_readme(**{**_args, "questions": []})
    except ValueError:
        return True
    return False

check("질문 없으면 ValueError", _no_question_raises)

✅ 통과 - 제목·요약
✅ 통과 - 절 순서
✅ 통과 - 데이터 표
✅ 통과 - 끝 줄바꿈 하나
✅ 통과 - 질문 없으면 ValueError


### AI 활용 공개 채점 기준

| 수준 | 기준 |
|---|---|
| 우수 | 단계별로 구체적, 채택·수정·검증 방법이 코드와 일치 |
| 보통 | 도구와 용도는 있으나 검증 방법이 막연함 |
| 미흡 | "참고함" 수준, 또는 코드와 불일치 |

---
## 한 학기 성찰 (필수)

다음 질문에 각각 3문장 이상으로 답하세요.

1. 1주차의 나와 비교해, **AI가 준 코드를 대하는 방식**은 어떻게 달라졌나요?
2. 이번 학기에 가장 기억에 남는 **버그나 잘못된 결론**과, 그것을 찾아낸 방법은?
3. 앞으로 다른 수업이나 일에서 AI를 쓸 때 지킬 **나만의 원칙 세 가지**는?

(학생 작성 영역)